In [3]:

import os
import shutil
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, array_to_img
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.metrics import classification_report


In [4]:
print(tf.__version__)

2.21.0


In [5]:
import os
import shutil
from sklearn.model_selection import train_test_split

base_dataset_dir = "dataset"
classes = ["bike", "car", "lorry","unknown"]

# Paths for new split folders
split_base_dir = "/content/split_dataset"
train_dir = os.path.join(split_base_dir, "train")
val_dir = os.path.join(split_base_dir, "val")
test_dir = os.path.join(split_base_dir, "test")

# Remove old splits if exist
if os.path.exists(split_base_dir):
    shutil.rmtree(split_base_dir)
os.makedirs(train_dir)
os.makedirs(val_dir)
os.makedirs(test_dir)

# Split images into train, val, test
for cls in classes:
    cls_path = os.path.join(base_dataset_dir, cls)
    images = [f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

    # Split: 20% test first
    train_val_imgs, test_imgs = train_test_split(images, test_size=0.2, random_state=42)
    # Split remaining 80% into 20% validation
    train_imgs, val_imgs = train_test_split(train_val_imgs, test_size=0.2, random_state=42)

    # Create class folders
    for folder, imgs in zip([train_dir, val_dir, test_dir], [train_imgs, val_imgs, test_imgs]):
        cls_folder = os.path.join(folder, cls)
        os.makedirs(cls_folder, exist_ok=True)
        for img in imgs:
            shutil.copy(os.path.join(cls_path, img), os.path.join(cls_folder, img))

print("Train/Validation/Test folders created successfully!")

Train/Validation/Test folders created successfully!


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

img_size = (224, 224)
batch_size = 32

# augmentation + preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    zoom_range=0.25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

# No augmentation for val/test
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Generators
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 1283 images belonging to 4 classes.
Found 322 images belonging to 4 classes.
Found 405 images belonging to 4 classes.


In [7]:
# Load the MobileNetV2 model with pre-trained weights
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet', alpha=0.35)


base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-15:]:
    if hasattr(layer, 'kernel_regularizer'):
        layer.kernel_regularizer = tf.keras.regularizers.l2(0.005)

# Create the model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.05)),  # Increased neurons
    Dropout(0.3),  # Reduced dropout
    Dense(train_gen.num_classes, activation='softmax')
])

# Compile the model with label smoothing
model.compile(
    optimizer=Adam(learning_rate=1e-4),  # Increased learning rate
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.01),  # Label smoothing to reduce overconfidence
    metrics=['accuracy']
)

2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [8]:
# Train model
epochs = 10
history = model.fit(train_gen, validation_data=val_gen, epochs=10, verbose=1)


Epoch 1/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 125s 3s/step - accuracy: 0.5885 - loss: 12.1095 - val_accuracy: 0.8292 - val_loss: 11.1063
Epoch 2/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 120s 2s/step - accuracy: 0.8714 - loss: 10.4949 - val_accuracy: 0.9006 - val_loss: 9.8463
Epoch 3/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 78s 2s/step - accuracy: 0.8815 - loss: 9.3632 - val_accuracy: 0.9224 - val_loss: 8.7502
Epoch 4/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 86s 2s/step - accuracy: 0.9088 - loss: 8.3134 - val_accuracy: 0.9224 - val_loss: 7.7699
Epoch 5/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.9283 - loss: 7.3879 - val_accuracy: 0.9348 - val_loss: 6.9048
Epoch 6/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.9408 - loss: 6.5463 - val_accuracy: 0.9410 - val_loss: 6.1177
Epoch 7/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.9353 - loss: 5.8164 - val_accuracy: 0.9472 - val_loss: 5.4128
Epoch 8/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 78s 2s/step - accuracy: 0.9509 - loss: 5.1636 - val_accuracy: 0.9503 - val_

In [9]:
test_loss, test_accuracy = model.evaluate(test_gen, verbose=1)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 914ms/step - accuracy: 0.9728 - loss: 3.7460
Test Accuracy: 97.28%


In [11]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt


y_true = test_gen.classes
y_pred = np.argmax(model.predict(test_gen), axis=-1)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=test_gen.class_indices.keys()))

# Confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=test_gen.class_indices.keys(),
            yticklabels=test_gen.class_indices.keys())
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 679ms/step
Classification Report:
              precision    recall  f1-score   support

        bike       0.99      0.99      0.99       101
         car       0.98      0.99      0.99       101
       lorry       0.95      0.98      0.97       102
     unknown       0.97      0.93      0.95       101

    accuracy                           0.97       405
   macro avg       0.97      0.97      0.97       405
weighted avg       0.97      0.97      0.97       405



NameError: name 'sns' is not defined

<Figure size 800x600 with 0 Axes>

In [12]:
model.save("vehicle_model.h5")

In [ ]:
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model(
    "vehicle_model.h5")
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_data_gen():
    for i in range(100):
        images, _ = next(train_gen)
        yield [images.astype(np.float32)]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

tflite_path = "vehicle_model_int8.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(" Quantized model saved at:", tflite_path)


INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmpfh1ctt3a\assets


INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmpfh1ctt3a\assets


Saved artifact at 'C:\Users\User\AppData\Local\Temp\tmpfh1ctt3a'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2425638893520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638892368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638892560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638892176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638893712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638893328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638891408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638891600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638892944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2425638893904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  24256

c:\Users\User\drive-sense\venv_model\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
